In [1]:
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from matplotlib.ticker import FuncFormatter
from matplotlib.dates import *

In [2]:
from config import DRIVER as driver

In [3]:
def to_list(cursor):
    return list(map(dict, cursor))

def to_data_frame(cursor):
    return pd.DataFrame(to_list(cursor))

def get_transactions(ind, am):
    url = "http://127.0.0.1:28081/get_outs"
    # Define the JSON payload
    payload = {"outputs":[{"amount": am, "index":ind}], "get_txid": True}
    # Define the headers
    headers = {
        "Content-Type": "application/json"
    }

    # Make the POST request
    response = requests.post(url, json=payload, headers=headers)
    return response.json()

First we will get the number of inputs with a mix in equal to 0. Remember this data has been creaed in a local tesnet, so the number with mixin equal to 0 may be elevated to show the privicy risk of using mixin 0. Rememebre that mixin 0 is not recommended for privacy reasons and you can no longer use it in the Monero main network.

In [4]:
with driver.session() as session:
    num_inputs = to_list(session.run(query=  """MATCH (i:Input) RETURN count(i)"""))[0]['count(i)']
    num_inputs_mix_0 = to_list(session.run(query=  """MATCH (i:Input) WHERE i.mixin = '0' RETURN count(i)"""))[0]['count(i)']

print(f"Number of inputs: {num_inputs}")
print(f"\nNumber of inputs with mixin equal to 0: {num_inputs_mix_0}. This inputs do not have the benefits of ring signatures")

Number of inputs: 320

Number of inputs with mixin equal to 0: 126. This inputs do not have the benefits of ring signatures


Our next step is to get all the information of those inputs with mixins equal to 0 to see what information we can get from them.

In [5]:
query = """MATCH (i:Input)-[:TX_INPUT]->(t:Transactions)
            WHERE i.mixin = '0'
            RETURN i, t"""

with driver.session() as session:
    data = to_list(session.run(query=query))
    
    rows = []
    for record in data:
        input_node = record['i']
        transaction_node = record['t']
        row = {
                    'input_mixin': input_node.get('mixin'),
                    'input_anonset': input_node.get('anonset'),
                    'input_id': input_node.get('id'),
                    'input_value': input_node.get('value'),
                    'key_offset': input_node.get('key_offset'),
                    'transaction_fee': transaction_node.get('fee'),
                    'transaction_id': transaction_node.get('id'),
                    'transaction_hash': transaction_node.get('hash')
                }
        rows.append(row)
inputs = pd.DataFrame(rows)
inputs

,input_mixin,input_anonset,input_id,input_value,key_offset,transaction_fee,transaction_id,transaction_hash
0,0,80,i0,500000000000,[1],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
1,0,80,i1,90000000000,[13],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
2,0,80,i2,10000000000000,[12],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
3,0,11,i3,2000000000,[3],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
4,0,80,i4,7000000000000,[6],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
...,...,...,...,...,...,...,...,...
121,0,93,i121,90000000000,[27],4556569224,t118,10ca75a51580e233b70aef08df952fb9efbf7e0f22df37...
122,0,108,i122,10000000000000,[28],2394344630,t119,516b7d7bbcb748d23aeea64459db1d688eff59ce783344...
123,0,89,i123,500000000000,[28],2394344630,t119,516b7d7bbcb748d23aeea64459db1d688eff59ce783344...
124,0,93,i124,90000000000,[28],2394344630,t119,516b7d7bbcb748d23aeea64459db1d688eff59ce783344...


Notice that there are key_offset that repeat. This is because the key_offset it point us at the output of some previous block, and that block can have multiple output. Is the key (which is an information will get on the next steps) along with the key_offset that will help us to identify the output that is being spent.

In [8]:
k_offsets_mix_0 = {}
tx_used = []
for _, row in inputs[['input_value', 'key_offset', 'transaction_hash']].iterrows():
    key_offset = row.key_offset.split('[')[1].split(']')[0]
    value = row.input_value
    outs = get_transactions(int(key_offset), int(value))
    if key_offset not in k_offsets_mix_0.keys():
        k_offsets_mix_0[key_offset] = [{'block': outs['outs'][0]['height'], 'key': outs['outs'][0]['key'], 'amount': value }]
    else:
        k_offsets_mix_0[key_offset].append({'block': outs['outs'][0]['height'], 'key': outs['outs'][0]['key'], 'amount': value })
    tx_used.append(outs['outs'][0]['key'])

With this last dictionary, we have the information of each key_offset used in the transactions with mixin equal to 0. As the mixins is 0, the transactions that appears is 100% the one being spend. If this key it is used in another transaction, we will know that it is not the real one as it has been previosuly spent.

In [9]:
query = """MATCH (i:Input)-[:TX_INPUT]->(t:Transactions)
            WHERE i.mixin <> '0'
            RETURN i, t"""

with driver.session() as session:
    data = to_list(session.run(query=query))
    
    rows = []
    for record in data:
        input_node = record['i']
        transaction_node = record['t']
        row = {
                    'input_mixin': input_node.get('mixin'),
                    'input_anonset': input_node.get('anonset'),
                    'input_id': input_node.get('id'),
                    'input_value': input_node.get('value'),
                    'key_offset': input_node.get('key_offset'),
                    'transaction_fee': transaction_node.get('fee'),
                    'transaction_id': transaction_node.get('id'),
                    'transaction_hash': transaction_node.get('hash')
                }
        rows.append(row)
inputs_mul_mix = pd.DataFrame(rows)
inputs_mul_mix

,input_mixin,input_anonset,input_id,input_value,key_offset,transaction_fee,transaction_id,transaction_hash
0,1,122,i126,10000000000000,"[27, 72]",2055711202,t133,2add8157b83055e2f5ad320370f8360f984562aa4c46cf...
1,1,381,i127,7000000000000,"[65, 90]",2000000000,t407,601c0e7424bfa708448346191685642c666082f8df83f0...
2,1,395,i128,10000000000000,"[112, 133]",2000000000,t408,b9bef1fb81268884c2809f043dc2bb2f66217104c8910a...
3,1,395,i129,10000000000000,"[202, 96]",2000000000,t409,7cd75f62b20bdd7a2b867e8dc2841b7c877354a6fea46f...
4,1,381,i130,7000000000000,"[130, 84]",2000000000,t410,0ddcb715b3059470f25ab087ebde5ace91edb5fefae515...
...,...,...,...,...,...,...,...,...
189,1,891,i315,7000000000000,"[140, 643]",2315322109,t967,9fac70abed0c324333754e6af91d32eb3f39dd75f335ff...
190,1,891,i316,7000000000000,"[208, 345]",2288711395,t968,c33ff4eeffe4933a59c43c003b90e4782cd563e22543fd...
191,1,891,i317,7000000000000,"[141, 420]",2288711395,t968,c33ff4eeffe4933a59c43c003b90e4782cd563e22543fd...
192,1,928,i318,10000000000000,"[117, 652]",2237573312,t970,ab8c9013b8bc0700edc167f0ba65cec5b732484c8717e7...


In [11]:
k_offsets = {} 
for _, row in inputs_mul_mix[['input_value', 'key_offset', 'transaction_hash']].iterrows():
    key_offset = row.key_offset.split('[')[1].split(']')[0].split(',')
    value = row.input_value
    key_i = 0
    for i, key in enumerate(key_offset):
        key_i = int(key)
        out = get_transactions(key_i, int(value))
        if out['outs'][0]['key'] in tx_used:
            print(key)
            print(out['outs'][0]['key'])
            num_mixin = len(key_offset)
            if num_mixin == 2:
                tx_used.append(get_transactions(int(key_offset[1-i]), int(value))['outs'][0]['key'])
            if str(num_mixin) in k_offsets.keys():
                k_offsets[str(num_mixin)] += 1
            else:
                k_offsets[str(num_mixin)] = 1
            print('-----------')
            print('Vul found')

 18
7809664420209cfefee25ff6a379c6aea77d25badadf485d54e014bce9c4fa00
-----------
Vul found
 15
82d5d7f3284a5db5ac3a0d9e14c962dedd1cc032c6c33f3910abded3e7a66fc6
-----------
Vul found
 7
737fe21df7580e3b2bada22c5313daf89e6d27e72b4cbed40469d6c92b41cf9a
-----------
Vul found
 1
274c137fe0fd5c1705281be0b06eddde310fa3b676f44e8ed8ba7dc61747064f
-----------
Vul found
118
8fef80d09346b1b1d4a79fe85b47ead451c1faaf7b56945d9a2f024757d19dd5
-----------
Vul found
 7
3ce626858931a75eaba2cc0d8816f1b28114b7ce6dcb3ad402af41cbef9db552
-----------
Vul found
 24
e4468c1540449b7ea51c45a1c749555c9648a3346a262c27d950142fb0535132
-----------
Vul found
118
8fef80d09346b1b1d4a79fe85b47ead451c1faaf7b56945d9a2f024757d19dd5
-----------
Vul found
 19
e7b61b835aee572f8fe789a913e8ec521a1b6215ab52169e9a18113aad765475
-----------
Vul found
 15
7de6615575ae6647ad00b43d78a8349923789e2a263dce1a19afd9e1f2687a58
-----------
Vul found
7
3ce626858931a75eaba2cc0d8816f1b28114b7ce6dcb3ad402af41cbef9db552
-----------
Vul found
 28


In [12]:
inputs['key_offset'].values

array(['[1]', '[13]', '[12]', '[3]', '[6]', '[15]', '[2]', '[11]', '[18]',
       '[10]', '[8]', '[5]', '[18]', '[4]', '[8]', '[18]', '[7]', '[4]',
       '[8]', '[11]', '[5]', '[7]', '[17]', '[2]', '[9]', '[10]', '[18]',
       '[6]', '[22]', '[21]', '[8]', '[20]', '[3]', '[14]', '[0]', '[4]',
       '[21]', '[11]', '[7]', '[14]', '[6]', '[0]', '[20]', '[16]',
       '[17]', '[7]', '[10]', '[21]', '[4]', '[2]', '[5]', '[9]', '[9]',
       '[17]', '[11]', '[5]', '[8]', '[21]', '[2]', '[15]', '[20]', '[0]',
       '[10]', '[4]', '[9]', '[9]', '[2]', '[5]', '[7]', '[17]', '[10]',
       '[12]', '[22]', '[19]', '[0]', '[1]', '[22]', '[15]', '[3]',
       '[14]', '[19]', '[0]', '[15]', '[13]', '[3]', '[13]', '[19]',
       '[1]', '[12]', '[20]', '[14]', '[12]', '[6]', '[22]', '[23]',
       '[16]', '[23]', '[23]', '[1]', '[13]', '[16]', '[1]', '[19]',
       '[3]', '[23]', '[6]', '[16]', '[25]', '[25]', '[25]', '[26]',
       '[26]', '[26]', '[25]', '[26]', '[24]', '[24]', '[24]', '[27]',


In [74]:
for k_off in k_offsets.keys():
    if k_off in k_offsets_mix_0.keys():
        print(k_offsets_mix_0[k_off])

[{'block': 28, 'key': '85734c35000ccb1c066037bc36176f62a540a09d8249a7b61a14d799e1954f42', 'amount': '7000000000000'}, {'block': 28, 'key': '408ad76aff19df7466246395b01ed17d876146c9087a513db796c3db97c403ee', 'amount': '500000000000'}, {'block': 28, 'key': 'c5b6d018b9bbdc0d6f109ec2ab7d67b7798a9bdc3133d0dbfcb2ecaf9e70f811', 'amount': '90000000000'}]
